# 🌤️ Extracting `Daily Weather Data` from the Meteostat API and Loading It into a Database

Learning Goals

By the end of this notebook, you should be able to:

- Understand how to connect to and extract data from an external API (Meteostat).

- Load raw JSON data into a PostgreSQL database using Python and SQLAlchemy.

- Appreciate how this fits into a real data pipeline (Extract → Load → Transform).

## 1. Context: Where We Are in the Pipeline

In analytics workflows, the first step is often data ingestion.
Before we can analyze, clean, or visualize, we need to get the data out of APIs and into a database.

Today, we’ll:

- Extract daily weather data for 3 major US airports.

- Store the raw JSON response as-is in our database (so it can be transformed later).

- Later, we’ll transform it into structured tables for analysis.

- Think of this as building the “E” and “L” in ELT.

The Goal of this Notebook is to get raw JSON data for Daily and Hourly Weather for 3 airport weather stations and load it as it is to our database.
- Find Station IDs for **defined** airports 
- Define the start and end of the period
- get the API Key from the `.env`

## 2. Prerequisites and Setup

We’ll need:

- An API key for Meteostat on RapidAPI.

- A `.env` file containing both API and database credentials.

- Libraries: requests, pandas, sqlalchemy, dotenv.

In [ ]:
# we will need the credentials we saved in the .env file
from dotenv import dotenv_values

# We also will need SQLAlchemy and its functions
from sqlalchemy import create_engine, types
from sqlalchemy.dialects.postgresql import JSON as postgres_json

import pandas as pd

# requests library will make the API calls. 
# the json package will parse the JSON string and convert it to Python data structures
import requests
import json

# with 'datetime' we want to catch the timestamp of the API call. For the actuality reference. 
# and 'time' for slowing down a .bit
from datetime import datetime
import time

### Defining Airports and finding the Station IDs

For our Pipeline we will use weather data from the weather stations at the 3 highly frequented airports
- **JFK**: John F. Kennedy Airport
- **MIA**: Miami International Airport
- **LAX**: Los Angeles Airport

To find the Station IDs for the airpors without stressing our API Call limits, we will use the   search option of the **https://meteostat.net/**  

We can search for the names of the airports above and find the Station IDs.

Let's add them to the dictionary below.

In [ ]:
airport_staids = {
    'JFK': '72503',   # John F. Kennedy International
    'MIA': '72202',   # Miami International
    'LAX': ???
           }

### Defining the period

Our flight Data is from 2024-01-01 until 2024-03-31. For the lectures we will use the same period for the meteostat JSON API.

In [ ]:
period_start = "2024-01-01"
period_end = "2024-03-31"

## 3. Load API and DB Credentials

In [ ]:

config = dotenv_values()

api_key = config['RAPIDAPI_KEY']  # align the key label with your .env file
pg_user = config['POSTGRES_USER']
pg_pass = config['POSTGRES_PASS']
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']


## 4. Building and Testing the API Call

Let’s test the querystring for one airport.

Each API call will get 3 months of weather data for one Station ID.  

In the [**RapidAPI**](https://rapidapi.com/meteostat/api/meteostat/playground) interface you can find the code syntax we need to make the call. 

For each call we need to create a querystring with required parameters.

In [ ]:
url = "https://meteostat.p.rapidapi.com/stations/daily"
headers = {
    "X-RapidAPI-Key": api_key,
    "X-RapidAPI-Host": "meteostat.p.rapidapi.com"
}

# Just to inspect the query for one airport
for airport in airport_staids:
    querystring = {
        "station": airport_staids[airport],
        "start": period_start,
        "end": period_end,
        "model": "true"  # use model-based data where necessary
    }
    print(airport, "\n", querystring)


### Objectives -  Daily Station Data:

- create a for-loop for the 3 airports, generating a **querystring for each API call**
- define an empty dictionary to collect: 
  - time of the call
  - airport code
  - station id
  - related data
- make the API calls using the for-loop and fill the dictionary
- create pandas dataframe from the dictionary
- load the DB credentials from the `.env`
- create the engine
- define data types for the postgresql table columns
- using pandas import the dataframe to the Table in the Schema of the DB

### API CALL daily (per station)

In [ ]:
#  let's catch each response in a dictionary. create an empty dictionary with the following keys:

weather_dict = {'extracted_at':[], 
                'airport_code':[], 
                'station_id':[], 
                'extracted_data':[]
               }


# for-loop for the querystrings
for airport in airport_staids:
    querystring = {
        "station":airport_staids[airport]
        ,"start":period_start
        ,"end":period_end
        ,"model":"true"
    }
    
    # making one call with the current querystring
    response = requests.get(url, headers=headers, params=querystring)

    # Add a slight delay to respect API limits
    time.sleep(1)
                
    # appending data to the dictionary:
    weather_dict['extracted_at'].append(???)       # timestamp, 
    weather_dict['airport_code'].append(???)       # airport code    
    weather_dict['station_id'].append(???)         # weater Station ID
    weather_dict['extracted_data'].append(json.loads(response.text))   # JSON string

## Preview the Extracted Data 

In [ ]:
weather_daily_df = pd.DataFrame(weather_dict)
weather_daily_df

Each row corresponds to:

- one API call

- one airport

- one JSON payload containing the weather data

⚠️ We’re not cleaning or flattening the JSON yet — that’s for the Transform step.

## 5. Load the Raw JSON into the Database

We’ll store it as raw data (no transformations yet).

Each record = one JSON blob.

Now all we need to create a table in your Schema in our database is part of the `weather_daily_df` dataframe.  
We can use pandas' ability to work with SQLAlchemy and "save" the data to the DB using the `.to_sql()` method

In [ ]:
# updating the url
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

# creating the engine
engine = create_engine(url, echo=False)

In [ ]:
engine.url # checking the url (pass is hidden)

In [ ]:
# defining data types for the DB
dtype_dict = {
    'extracted_at':types.DateTime,
    'airport_code': types.String,
    'station_id': types.Integer,
    'extracted_data':postgres_json
             }

In [ ]:
# writing dataframe to DB
weather_daily_df.to_sql(name = 'weather_daily_raw', 
                       con = engine, 
                       schema = pg_schema, # pandas is allowing to specify, in which schema the table shall be created
                       if_exists='replace', 
                       dtype=dtype_dict,
                       index=False
                      )

✅ You should see an output showing the number of rows written (should be 3).

Now check in DBeaver — you should see a new table:
`your_schema.weather_daily_raw`

-----------

## 💭 Wrap-up: What You Just Did

✅ You’ve completed the Extract and Load stages of a real-world ELT pipeline.

| Step    | Description                     | Tool                  |
| ------- | ------------------------------- | --------------------- |
| Extract | Called external API (Meteostat) | `requests`            |
| Load    | Saved raw JSON to PostgreSQL    | `pandas + SQLAlchemy` |


## 🚀 Next Steps (Transformation Preview)

In the next chapters, you’ll:

- Flatten the JSON data.

- Create structured tables (e.g. daily temperature, precipitation).

- Link it with flight data for analysis.


------------


## For the Curious: Quick Peek at the JSON Structure
Not part of the pipeline — just exploration



In [ ]:
sample = weather_daily_df['extracted_data'][0]
print(json.dumps(sample, indent=2)[:1000])

In [ ]:
# using pd.json_normalize() twice to get to the weather_stats of one airport under 'data'

df_JFK = pd.json_normalize(pd.json_normalize(weather_daily_df['extracted_data']).loc[0, 'data'])
df_JFK

#  compare if needed to the JSON for 'JFK', first row 
weather_daily_df.loc[0,'extracted_data']